In [22]:
import csv
import os
import math
from sklearn import linear_model
from sklearn.metrics import mean_squared_error, mean_absolute_error

class MyLinearUnivariateRegression:
    def __init__(self):
        self.intercept_ = 0.0
        self.coef_ = 0.0

    def fit(self, x, y):
        sx = sum(x)
        sy = sum(y)
        sx2 = sum(i * i for i in x)
        sxy = sum(i * j for (i, j) in zip(x, y))
        n = len(x)

        w1 = (n * sxy - sx * sy) / (n * sx2 - sx * sx)
        w0 = (sy - w1 * sx) / n
        self.intercept_, self.coef_ = w0, w1

    def predict(self, x):
        if isinstance(x[0], list):
            return [self.intercept_ + self.coef_ * val[0] for val in x]
        else:
            return [self.intercept_ + self.coef_ * val for val in x]

class MyLinearBivariateRegression:
    def __init__(self):
        self.w0 = 0.0
        self.w1 = 0.0
        self.w2 = 0.0

    def _det3x3(self, m):
        return (m[0][0] * (m[1][1] * m[2][2] - m[1][2] * m[2][1]) -
                m[0][1] * (m[1][0] * m[2][2] - m[1][2] * m[2][0]) +
                m[0][2] * (m[1][0] * m[2][1] - m[1][1] * m[2][0]))

    def fit(self, X, Y):
        n = len(Y)
        if n == 0: return
        sum_x1 = sum(x[0] for x in X)
        sum_x2 = sum(x[1] for x in X)
        sum_y = sum(Y)
        sum_x1_sq = sum(x[0]**2 for x in X)
        sum_x2_sq = sum(x[1]**2 for x in X)
        sum_x1_x2 = sum(x[0]*x[1] for x in X)
        sum_x1_y = sum(x[0]*y for x, y in zip(X, Y))
        sum_x2_y = sum(x[1]*y for x, y in zip(X, Y))

        eps = 1e-4

        A00 = n + eps
        A11 = sum_x1_sq + eps
        A22 = sum_x2_sq + eps

        mat_A = [
            [A00, sum_x1, sum_x2],
            [sum_x1, A11, sum_x1_x2],
            [sum_x2, sum_x1_x2, A22]
        ]
        det_A = self._det3x3(mat_A)

        mat_w0 = [
            [sum_y, sum_x1, sum_x2],
            [sum_x1_y, A11, sum_x1_x2],
            [sum_x2_y, sum_x1_x2, A22]
        ]

        mat_w1 = [
            [A00, sum_y, sum_x2],
            [sum_x1, sum_x1_y, sum_x1_x2],
            [sum_x2, sum_x2_y, A22]
        ]

        mat_w2 = [
            [A00, sum_x1, sum_y],
            [sum_x1, A11, sum_x1_y],
            [sum_x2, sum_x1_x2, sum_x2_y]
        ]

        if det_A == 0:
            return

        self.w0 = self._det3x3(mat_w0) / det_A
        self.w1 = self._det3x3(mat_w1) / det_A
        self.w2 = self._det3x3(mat_w2) / det_A

    def predict(self, X):
        return [self.w0 + self.w1 * x[0] + self.w2 * x[1] for x in X]

def calculate_manual_metrics(real, computed):
    error_mae = sum(abs(r - c) for r, c in zip(real, computed)) / len(real)
    error_rmse = math.sqrt(sum((r - c) ** 2 for r, c in zip(real, computed)) / len(real))
    return error_mae, error_rmse

def loadDataMultipleFeatures(fileName, inputVariabNames, outputVariabName):
    inputs = []
    outputs = []

    with open(fileName) as csv_file:
        csv_reader = csv.reader(csv_file, delimiter=',')
        dataNames = next(csv_reader)

        try:
            in_indexes = [dataNames.index(name) for name in inputVariabNames]
            out_index = dataNames.index(outputVariabName)
        except ValueError:
            print(f"Coloanele nu au putut fi găsite în {fileName}.")
            return [], []

        for row in csv_reader:
            try:
                in_vals = [float(row[idx]) for idx in in_indexes]
                out_val = float(row[out_index])
                inputs.append(in_vals)
                outputs.append(out_val)
            except (ValueError, IndexError):
                continue

    return inputs, outputs

def solve_regression_problem(file_name):
    print(f"\n--- Analizăm fișierul: {file_name} ---")
    crtDir = 'C:/Laborator AI/Laborator 5/'
    filePath = os.path.join(crtDir, file_name)

    if not os.path.exists(filePath):
        print(f"Fișierul {filePath} nu a fost găsit!")
        return

    # A) Univariat: "Family" -> "Happiness"
    print("\n1. Regresie Univariată (Family -> Happiness)")
    inputs, outputs = loadDataMultipleFeatures(filePath, ['Family'], 'Happiness.Score')
    inputs_flat = [x[0] for x in inputs]

    split_idx = int(0.8 * len(inputs_flat))
    train_in, val_in = inputs_flat[:split_idx], inputs_flat[split_idx:]
    train_out, val_out = outputs[:split_idx], outputs[split_idx:]

    tool_reg_univ = linear_model.LinearRegression()
    tool_reg_univ.fit([[x] for x in train_in], train_out)
    pred_fam_tool = tool_reg_univ.predict([[x] for x in val_in])
    print(f"  Tool (sklearn) -> MAE: {mean_absolute_error(val_out, pred_fam_tool):.4f}, RMSE: {math.sqrt(mean_squared_error(val_out, pred_fam_tool)):.4f}")

    my_reg_univ = MyLinearUnivariateRegression()
    my_reg_univ.fit(train_in, train_out)
    pred_univ_my = my_reg_univ.predict(val_in)

    mae_my, rmse_my = calculate_manual_metrics(val_out, pred_univ_my)
    print(f"  Cod propriu -> MAE: {mae_my:.4f}, RMSE: {rmse_my:.4f}")

    # B) Bivariat: "GDP" + "Freedom" -> "Happiness"
    print("\n2. Regresie Bivariată (GDP + Freedom -> Happiness)")
    inputs_biv, outputs_biv = loadDataMultipleFeatures(
        filePath, ['Economy..GDP.per.Capita.', 'Freedom'], 'Happiness.Score'
    )

    split_idx_biv = int(0.8 * len(inputs_biv))
    train_in_biv, val_in_biv = inputs_biv[:split_idx_biv], inputs_biv[split_idx_biv:]
    train_out_biv, val_out_biv = outputs_biv[:split_idx_biv], outputs_biv[split_idx_biv:]

    # Tool (Sklearn)
    tool_reg_biv = linear_model.LinearRegression()
    tool_reg_biv.fit(train_in_biv, train_out_biv)
    pred_biv_tool = tool_reg_biv.predict(val_in_biv)

    rmse_tool = math.sqrt(mean_squared_error(val_out_biv, pred_biv_tool))
    mae_tool = mean_absolute_error(val_out_biv, pred_biv_tool)
    print(f"  Tool (sklearn) -> MAE: {mae_tool:.4f}, RMSE: {rmse_tool:.4f}")

    # Cod Propriu (MyLinearBivariateRegression)
    my_reg_biv = MyLinearBivariateRegression()
    my_reg_biv.fit(train_in_biv, train_out_biv)
    pred_biv_my = my_reg_biv.predict(val_in_biv)

    mae_biv_my, rmse_biv_my = calculate_manual_metrics(val_out_biv, pred_biv_my)
    print(f"  Cod propriu    -> MAE: {mae_biv_my:.4f}, RMSE: {rmse_biv_my:.4f}")

solve_regression_problem('v1_world-happiness-report-2017.csv')
solve_regression_problem('v2_world-happiness-report-2017.csv')
solve_regression_problem('v3_world-happiness-report-2017.csv')


--- Analizăm fișierul: v1_world-happiness-report-2017.csv ---

1. Regresie Univariată (Family -> Happiness)
  Tool (sklearn) -> MAE: 1.9729, RMSE: 2.0077
  Cod propriu -> MAE: 1.9729, RMSE: 2.0077

2. Regresie Bivariată (GDP + Freedom -> Happiness)
  Tool (sklearn) -> MAE: 0.6248, RMSE: 0.7863
  Cod propriu    -> MAE: 0.6248, RMSE: 0.7863

--- Analizăm fișierul: v2_world-happiness-report-2017.csv ---

1. Regresie Univariată (Family -> Happiness)
  Tool (sklearn) -> MAE: 0.9686, RMSE: 1.1156
  Cod propriu -> MAE: 0.9686, RMSE: 1.1156

2. Regresie Bivariată (GDP + Freedom -> Happiness)
  Tool (sklearn) -> MAE: 0.8136, RMSE: 0.9181
  Cod propriu    -> MAE: 0.8106, RMSE: 0.9168

--- Analizăm fișierul: v3_world-happiness-report-2017.csv ---

1. Regresie Univariată (Family -> Happiness)
  Tool (sklearn) -> MAE: 0.9686, RMSE: 1.1156
  Cod propriu -> MAE: 0.9686, RMSE: 1.1156

2. Regresie Bivariată (GDP + Freedom -> Happiness)
  Tool (sklearn) -> MAE: 0.6106, RMSE: 0.7736
  Cod propriu    -> 